# Radio Map Comparison: CNN-MLP vs Diffusion

Generates a full radio map from the trained **CNN-MLP** model by sampling a dense grid of RX positions, then compares against:
- Ground truth (from Sionna-RT simulation)
- **RadioLunaDiff** diffusion model output

**Sections:**
1. CNN-MLP grid prediction + interactive scene explorer
2. Diffusion model inference (requires pretrained checkpoints + data paths)
3. Side-by-side comparison + metrics

In [ ]:
import sys
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from tqdm.auto import tqdm

# Notebook lives in RadioLunaLink/ — parent is software/
SOFT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(SOFT_ROOT / "RadioLunaLink"))
sys.path.insert(0, str(SOFT_ROOT / "RadioLunaDiff"))

from models.cnn_mlp import CNNMLPModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## Configuration
Edit the cells below to point to your data and checkpoints.

In [ ]:
# ── RadioLunaLink data ─────────────────────────────────────────────────────────
DATA_ROOT  = str(SOFT_ROOT.parent.parent / "NASA_DCGR_NETWORKING/radio_data_2/radio_data_2")
FREQUENCY  = "415"          # "415" or "58"
SCENE_IDX  = 0              # terrain scene (0 – 499)
TX_IDX     = 0              # TX within that scene
GRID_BATCH = 2048           # RX positions per forward pass

# ── Model selection ───────────────────────────────────────────────────────────
MODEL_NAME  = "vit"                   # "cnn_mlp" or "vit"
CNNMLP_CKPT = str(SOFT_ROOT / "RadioLunaLink/checkpoints/cnn_mlp_best.pt")
VIT_CKPT    = str(SOFT_ROOT / "RadioLunaLink/checkpoints/vit_best.pt")

# ViT hyperparameters — must match what was used during training
VIT_PATCH_SIZE = 32   # default 16; use 32 for the smaller experiment
VIT_LAYERS     = 4    # default 6;  use 4 for the smaller experiment
VIT_HEADS      = 8
VIT_EMBED_DIM  = 256

# ── Diffusion model paths ─────────────────────────────────────────────────────
DIFF_MODEL_PATH   = str(SOFT_ROOT / "RadioLunaDiff/pretrained_models_network/diffusion")
K2_MODEL_PATH     = str(SOFT_ROOT / "RadioLunaDiff/pretrained_models_network/k2unet/best_k2_model.pth")
PMNET_MODEL_PATH  = str(SOFT_ROOT / "RadioLunaDiff/pretrained_models_network/pmnet/best_pm_model.pt")
DIFF_DATA_DIR1    = ""      # /path/to/refraction_dataset_fin1
DIFF_DATA_DIR2    = ""      # /path/to/refraction_dataset_fin2
NUM_DIFF_STEPS    = 4

# ── Vis settings ──────────────────────────────────────────────────────────────
RM_MIN, RM_MAX = -200, 10
ERR_CLIM = 30

---
## Part 1 — CNN-MLP Radio Map Generation

In [ ]:
from models.cnn_mlp import CNNMLPModel
from models.vit_attention import CoordCondViT

def load_model(model_name, device):
    if model_name == "cnn_mlp":
        m = CNNMLPModel()
        ckpt_path = CNNMLP_CKPT
    elif model_name == "vit":
        m = CoordCondViT(
            patch_size=VIT_PATCH_SIZE,
            n_layers=VIT_LAYERS,
            n_heads=VIT_HEADS,
            embed_dim=VIT_EMBED_DIM,
        )
        ckpt_path = VIT_CKPT
    else:
        raise ValueError(model_name)
    m.load_state_dict(torch.load(ckpt_path, map_location=device))
    m.eval().to(device)
    print(f"Loaded {model_name} from {ckpt_path}")
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  {n:,} parameters")
    return m

model = load_model(MODEL_NAME, DEVICE)

In [ ]:
def load_scene(data_root, frequency, scene_idx, tx_idx):
    """Load heightmap, TX coord, and ground-truth radio map for one (scene, TX) pair."""
    root = Path(data_root)

    hm_raw  = np.load(root / "hm" / f"hm_{scene_idx}.npy").astype(np.float32)
    H, W    = hm_raw.shape[:2]
    hm_norm = (hm_raw - hm_raw.mean()) / (hm_raw.std() + 1e-6)

    tx_map   = np.load(root / "tx" / f"tx_{scene_idx}_{tx_idx}.npy")
    tx_loc   = np.argwhere(tx_map != 0)[0]
    tx_row, tx_col = int(tx_loc[0]), int(tx_loc[1])
    tx_coord = np.array([tx_col / W, tx_row / H], dtype=np.float32)

    rm_gt = np.load(root / f"rm{frequency}" / f"rm_{scene_idx}_{tx_idx}.npy").astype(np.float32)
    rm_gt = np.where(np.isnan(rm_gt), -200.0, rm_gt)

    return hm_raw, hm_norm, tx_coord, tx_row, tx_col, rm_gt, H, W


def _build_rx_grid(H, W):
    """All pixel positions as normalised (col/W, row/H) RX coords, shape (H*W, 2)."""
    gr, gc = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
    return np.stack([gc.flatten() / W, gr.flatten() / H], axis=1).astype(np.float32)


def predict_radio_map(model, hm_norm, tx_coord, H, W, batch_size, device):
    """Predict signal strength at every pixel, returning an (H, W) map.

    Caches the expensive per-scene computation (CNN backbone or ViT encoder)
    so it only runs once, then loops only the cheap per-RX-position parts.
    """
    rx_coords = _build_rx_grid(H, W)                                        # (H*W, 2)
    hm_t  = torch.from_numpy(hm_norm[np.newaxis, np.newaxis]).to(device)   # (1,1,H,W)
    tx_t  = torch.from_numpy(tx_coord).unsqueeze(0).to(device)             # (1,2)

    with torch.no_grad():
        if isinstance(model, CoordCondViT):
            # Cache: patch embedding + full transformer encoder + TX query
            tokens  = model.patch_embed(hm_t)           # (1, N, embed_dim)
            encoded = model.encoder(tokens)              # (1, N, embed_dim)
            tx_q    = model._encode_coord(tx_t)          # (1, embed_dim)

            def _predict_batch(rx_batch):
                B     = len(rx_batch)
                rx_q  = model._encode_coord(rx_batch)                         # (B, embed_dim)
                query = (tx_q.expand(B, -1) + rx_q).unsqueeze(1)             # (B, 1, embed_dim)
                att   = model.cross_attn(query, encoded.expand(B, -1, -1))   # (B, 1, embed_dim)
                return model.head(att.squeeze(1)).squeeze(1)                  # (B,)
        else:
            # CNN-MLP: cache backbone + TX coord encoder
            map_feat = model.backbone(hm_t)              # (1, embed_dim)
            tx_feat  = model.coord_enc(tx_t)             # (1, embed_dim)

            def _predict_batch(rx_batch):
                B          = len(rx_batch)
                rx_feat    = model.coord_enc(rx_batch)
                coord_feat = tx_feat.expand(B, -1) + rx_feat
                fused      = torch.cat([map_feat.expand(B, -1), coord_feat], dim=1)
                return model.head(fused).squeeze(1)

    preds = []
    with torch.no_grad():
        for i in tqdm(range(0, len(rx_coords), batch_size), desc="Predicting grid", leave=False):
            rx_batch = torch.from_numpy(rx_coords[i:i+batch_size]).to(device)
            preds.append(_predict_batch(rx_batch).cpu().numpy())

    return np.concatenate(preds).reshape(H, W)

In [ ]:
hm_raw, hm_norm, tx_coord, tx_row, tx_col, rm_gt, H, W = load_scene(
    DATA_ROOT, FREQUENCY, SCENE_IDX, TX_IDX
)
rm_pred = predict_radio_map(cnnmlp, hm_norm, tx_coord, H, W, GRID_BATCH, DEVICE)

print(f"Scene {SCENE_IDX}, TX {TX_IDX}  ({W}×{H} px)")
print(f"TX position: row={tx_row}, col={tx_col}")
print(f"GT   range: [{rm_gt.min():.1f}, {rm_gt.max():.1f}] dBm")
print(f"Pred range: [{rm_pred.min():.1f}, {rm_pred.max():.1f}] dBm")

In [ ]:
def plot_cnnmlp_comparison(hm_raw, rm_gt, rm_pred, tx_row, tx_col,
                            rm_min=RM_MIN, rm_max=RM_MAX, err_clim=ERR_CLIM,
                            title=""):
    valid = rm_gt > -199
    rmse  = float(np.sqrt(np.mean((rm_pred[valid] - rm_gt[valid])**2)))
    mae   = float(np.mean(np.abs(rm_pred[valid] - rm_gt[valid])))
    err   = rm_pred - rm_gt

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    fig.suptitle(
        f"{title}  |  RMSE = {rmse:.2f} dBm   MAE = {mae:.2f} dBm   (valid pixels)",
        fontsize=13
    )

    # Heightmap
    axes[0].imshow(hm_raw, cmap='terrain')
    axes[0].plot(tx_col, tx_row, 'r*', ms=10)
    axes[0].set_title('Heightmap')
    axes[0].axis('off')

    # Ground truth
    im1 = axes[1].imshow(rm_gt, cmap='viridis', vmin=rm_min, vmax=rm_max)
    axes[1].plot(tx_col, tx_row, 'r*', ms=10)
    axes[1].set_title('Ground Truth')
    axes[1].axis('off')
    plt.colorbar(im1, ax=axes[1], fraction=0.046, label='dBm')

    # CNN-MLP prediction
    im2 = axes[2].imshow(rm_pred, cmap='viridis', vmin=rm_min, vmax=rm_max)
    axes[2].plot(tx_col, tx_row, 'r*', ms=10)
    axes[2].set_title('CNN-MLP Prediction')
    axes[2].axis('off')
    plt.colorbar(im2, ax=axes[2], fraction=0.046, label='dBm')

    # Error
    im3 = axes[3].imshow(err, cmap='RdBu_r', vmin=-err_clim, vmax=err_clim)
    axes[3].plot(tx_col, tx_row, 'k*', ms=10)
    axes[3].set_title('Error (Pred − GT)')
    axes[3].axis('off')
    plt.colorbar(im3, ax=axes[3], fraction=0.046, label='dBm')

    plt.tight_layout()
    plt.show()
    return rmse, mae


rmse_cnn, mae_cnn = plot_cnnmlp_comparison(
    hm_raw, rm_gt, rm_pred, tx_row, tx_col,
    title=f"Scene {SCENE_IDX}, TX {TX_IDX}"
)

### Interactive Scene Explorer
Use the sliders to explore different scene / TX combinations without re-running cells.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    scene_slider  = widgets.IntSlider(value=SCENE_IDX, min=0, max=499, step=1,
                                      description='Scene:', continuous_update=False)
    tx_slider     = widgets.IntSlider(value=TX_IDX, min=0, max=49, step=1,
                                      description='TX:', continuous_update=False)
    model_toggle  = widgets.ToggleButtons(options=["cnn_mlp", "vit"], value=MODEL_NAME,
                                          description='Model:')
    out = widgets.Output()

    def on_change(_):
        with out:
            out.clear_output(wait=True)
            try:
                m = load_model(model_toggle.value, DEVICE)
                hm_, hm_n_, tx_c_, tr_, tc_, rm_g_, H_, W_ = load_scene(
                    DATA_ROOT, FREQUENCY, scene_slider.value, tx_slider.value
                )
                rm_p_ = predict_radio_map(m, hm_n_, tx_c_, H_, W_, GRID_BATCH, DEVICE)
                plot_cnnmlp_comparison(
                    hm_, rm_g_, rm_p_, tr_, tc_,
                    title=f"{model_toggle.value}  —  Scene {scene_slider.value}, TX {tx_slider.value}"
                )
            except FileNotFoundError as exc:
                print(f"Missing data: {exc}")
            except Exception as exc:
                print(f"Error: {exc}")

    scene_slider.observe(on_change, names='value')
    tx_slider.observe(on_change, names='value')
    model_toggle.observe(on_change, names='value')
    display(widgets.VBox([model_toggle, scene_slider, tx_slider, out]))

except ImportError:
    print("ipywidgets not installed — run: pip install ipywidgets")

---
## Part 2 — Diffusion Model Comparison

Runs the full RadioLunaDiff pipeline (K²-UNet → PMNet → residual diffusion) on a test sample and overlays results.

> **Prerequisites:** Set `DIFF_DATA_DIR1`, `DIFF_DATA_DIR2`, and checkpoint paths in the Config cell. The cell below will skip gracefully if any path is missing.

In [ ]:
diff_ready = False

missing = [p for p in [DIFF_DATA_DIR1, DIFF_DATA_DIR2] if not p]
if missing:
    print("⚠  Diffusion data dirs not configured — skipping Part 2.")
    print("   Set DIFF_DATA_DIR1 and DIFF_DATA_DIR2 in the Config cell.")
else:
    try:
        from diffusers import UNet2DModel, DDPMScheduler
        from torchvision.transforms import Normalize
        from torch.utils.data import DataLoader, ConcatDataset
        from dataset.refraction_loaders_mhz import RefractLunarLoader, RefractLunarLoader2
        from k2net.diff_modules import K2_UNet
        from pmnet.models.pmnet_v3 import PMNet

        # ── Load diffusion model ───────────────────────────────────────────────
        diff_model = UNet2DModel.from_pretrained(DIFF_MODEL_PATH).to(DEVICE)
        diff_model.eval()

        # ── Load K²-UNet ──────────────────────────────────────────────────────
        k2_model = K2_UNet(inputs=4, k2_bin=True).to(DEVICE)
        k2_model.load_state_dict(torch.load(K2_MODEL_PATH, map_location=DEVICE))
        k2_model.eval()

        # ── Load PMNet ────────────────────────────────────────────────────────
        pmnet = PMNet(n_blocks=[3, 3, 27, 3], atrous_rates=[6, 12, 18],
                      multi_grids=[1, 2, 4], output_stride=8, in_ch=5).to(DEVICE)
        pmnet.load_state_dict(torch.load(PMNET_MODEL_PATH, map_location=DEVICE))
        pmnet.eval()

        print("All diffusion models loaded.")
        diff_ready = True

    except Exception as exc:
        print(f"Could not load diffusion models: {exc}")

In [ ]:
# ── Build test dataset and pick one sample ─────────────────────────────────
DIFF_SAMPLE_IDX = 0     # index into the test split — change to explore

if diff_ready:
    loader_args = dict(
        train_split_ratio=0.7, val_split_ratio=0.15,
        use_los_input=False, use_pathloss_input=False,
        non_global_norm=False, filt_hm=False, use_filt_and_norm=True,
        los_weight=0.2, crop_resize=False, tx_array=False, heavy_aug=False,
    )
    normalize_11 = Normalize(mean=[0.5], std=[0.5])   # [0,1] → [-1,1]

    class _Wrapper(torch.utils.data.Dataset):
        def __init__(self, ds):
            self.ds = ds
        def __len__(self): return len(self.ds)
        def __getitem__(self, i):
            inputs, gain, name = self.ds[i]
            k2_in  = inputs.clone()
            pm_in  = inputs.clone()
            return normalize_11(inputs), normalize_11(gain), name, pm_in, k2_in

    test_ds = _Wrapper(ConcatDataset([
        RefractLunarLoader(phase="test", root_dir=DIFF_DATA_DIR1, rm_path=f"rm{FREQUENCY}",
                           freq=(0 if FREQUENCY=="415" else 1), k2=False,
                           k2_path=f"k2_bin_{FREQUENCY}", num_original_maps_total=500, **loader_args),
        RefractLunarLoader2(phase="test", root_dir=DIFF_DATA_DIR2, rm_path=f"rm{FREQUENCY}",
                            freq=(0 if FREQUENCY=="415" else 1), k2=False,
                            k2_path=f"k2_bin_{FREQUENCY}", num_original_maps_total=500, **loader_args),
    ]))
    print(f"Diffusion test set size: {len(test_ds)}  (using sample {DIFF_SAMPLE_IDX})")
else:
    print("Skipped — diffusion models not available.")

In [ ]:
if diff_ready:
    cond, target, name, pm_inputs, k2_inputs = test_ds[DIFF_SAMPLE_IDX]
    cond      = cond.unsqueeze(0).to(DEVICE)
    target    = target.unsqueeze(0).to(DEVICE)
    pm_inputs = pm_inputs.unsqueeze(0).to(DEVICE)
    k2_inputs = k2_inputs.unsqueeze(0).to(DEVICE)

    scheduler = DDPMScheduler(
        num_train_timesteps=1000,
        beta_schedule="squaredcos_cap_v2",
        prediction_type="v_prediction"
    )
    scheduler.set_timesteps(NUM_DIFF_STEPS)

    with torch.no_grad():
        k2_pred = torch.sigmoid(k2_model(k2_inputs))
        pm_pred = pmnet(torch.cat([pm_inputs, k2_pred.clone()], dim=1))
        pm_pred_n = normalize_11(pm_pred)
        k2_map_n  = normalize_11(k2_pred)

        noise = torch.randn_like(target)
        for t in tqdm(scheduler.timesteps, desc="Diffusion steps", leave=False):
            model_in = torch.cat([noise, cond, pm_pred_n, k2_map_n], dim=1)
            noise_pred = diff_model(model_in, t, return_dict=False)[0]
            noise = scheduler.step(noise_pred, t, noise, return_dict=False)[0]

        diff_final = pm_pred_n + noise  # PMNet init + residual

    def _dn(t):  # [-1,1] → [0,1]
        return ((t.clamp(-1, 1) + 1) / 2).squeeze().cpu().numpy()

    diff_gt_01   = _dn(target)
    diff_pred_01 = _dn(diff_final)
    pm_pred_01   = _dn(pm_pred_n)
    k2_01        = _dn(k2_map_n)

    # Convert [0,1] → dBm for metric computation (global: 0=−200 dBm, 1=10 dBm)
    diff_gt_dbm   = diff_gt_01   * 210 - 200
    diff_pred_dbm = diff_pred_01 * 210 - 200

    valid_d = diff_gt_01 > 0.005  # ≈ > −199 dBm
    rmse_diff = float(np.sqrt(np.mean((diff_pred_dbm[valid_d] - diff_gt_dbm[valid_d])**2)))
    mae_diff  = float(np.mean(np.abs(diff_pred_dbm[valid_d] - diff_gt_dbm[valid_d])))
    print(f"Diffusion RMSE = {rmse_diff:.2f} dBm   MAE = {mae_diff:.2f} dBm  (sample: {name})")

In [ ]:
if diff_ready:
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle(
        f"RadioLunaDiff  |  RMSE = {rmse_diff:.2f} dBm   MAE = {mae_diff:.2f} dBm  \n"
        f"Sample: {name}   ({NUM_DIFF_STEPS} diffusion steps)",
        fontsize=12
    )

    # Row 0: intermediate outputs
    axes[0,0].imshow(k2_01, cmap='gray');         axes[0,0].set_title('K²-UNet prediction')
    axes[0,1].imshow(diff_pred_01 - pm_pred_01,
                     cmap='inferno', vmin=-0.3, vmax=0.3)
    axes[0,1].set_title('Predicted residual (Stage 3)')
    axes[0,2].imshow(diff_gt_01 - pm_pred_01,
                     cmap='inferno', vmin=-0.3, vmax=0.3)
    axes[0,2].set_title('GT residual')

    # Row 1: radio maps
    axes[1,0].imshow(diff_gt_01,   cmap='viridis', vmin=0, vmax=1); axes[1,0].set_title('Ground Truth')
    im = axes[1,1].imshow(diff_pred_01, cmap='viridis', vmin=0, vmax=1); axes[1,1].set_title('Diffusion prediction')
    axes[1,2].imshow(pm_pred_01,   cmap='viridis', vmin=0, vmax=1); axes[1,2].set_title('PMNet (Stage 2)')

    for ax in axes.flat: ax.axis('off')
    plt.tight_layout()
    plt.show()

---
## Part 3 — Side-by-Side Metrics Summary

In [ ]:
print("═" * 50)
print(f"{'Model':<20} {'RMSE (dBm)':>12} {'MAE (dBm)':>12}")
print("─" * 50)
print(f"{'CNN-MLP':<20} {rmse_cnn:>12.2f} {mae_cnn:>12.2f}")
if diff_ready:
    print(f"{'Diffusion':<20} {rmse_diff:>12.2f} {mae_diff:>12.2f}")
    print()
    better = "CNN-MLP" if rmse_cnn < rmse_diff else "Diffusion"
    delta  = abs(rmse_cnn - rmse_diff)
    print(f"  {better} wins by {delta:.2f} dBm RMSE")
    print()
    print("Note: CNN-MLP and Diffusion may be evaluated on different scenes.")
    print("For a fair comparison, align SCENE_IDX / DIFF_SAMPLE_IDX to the same scene.")
print("═" * 50)